# 02. 데이터 전처리 및 분석 마트 구축·검증

## 개요

- 목적: `docs/metrics.md`의 처리 방침을 집행해 `mart_session`, `mart_session_product`, 구매 여정용 `mart_user_product_session`을 생성하고 raw(`events`)에서 독립 계산한 값과 대조 검증한다.
- 방침 원천: `docs/metrics.md`(단일 원천). 이 노트북은 방침을 변경하지 않는다.
- 분석 단위: `mart_session`은 세션 1행, 나머지 두 마트는 유효 세션·상품 조합 1행이다. 구매 여정 마트는 remove-only 조합과 이벤트 시각을 추가로 보존한다.
- 순차 기준: 대표값은 strict(`event_time < event_time`)이며, inclusive(`≤`)는 동일 시각 민감도만 기록한다.
- 검증: 행수·유형별 카운트·revenue·순차 플래그를 raw 독립 계산값과 대조하고, 플래그 포함 관계와 복합키를 확인한다. 불일치 시 다음 단계로 진행하지 않는다.

### 전처리 방침 집행과 검증

01에서 확정한 품질 처리 방침을 raw에 직접 반영하지 않고, 마트 생성 SELECT에서 적용한다. 이벤트 단위 raw를 세션 및 세션·상품 단위로 변환하고 순차 퍼널용 파생변수를 생성한 뒤, raw 독립 계산값과의 일치 여부를 다음 단계 진입 게이트로 사용한다.

| 01 진단 결과 | 02 전처리·변환 | 검증 방식 |
|---|---|---|
| `user_session` NULL | 세션 분석 대상에서 제외 | raw 유효 세션 수와 마트 행수 대조 |
| 세션당 사용자 2명 이상 | 해당 세션 제외 | 세션 기본키와 raw 독립 행수 확인 |
| 지속시간 1일 초과 | 해당 세션 제외 | 동일 유효 세션 조건을 raw에서 독립 계산 |
| 음수·0원 가격 | 음수는 카운트·revenue에서 제외하고 0원은 카운트에만 포함 | 이벤트 유형별 합계와 revenue 대조 |
| 이벤트 단위 raw | 세션 1행, 세션·상품 조합 1행으로 집계 | 기본키·복합키와 집계 행수 확인 |
| 이벤트 발생 시각 | strict 순차 플래그 생성 | raw의 다른 계산식으로 플래그 합계 대조 |

따라서 이 노트북의 마트 구축은 데이터 정제, 분석 단위 변환, 파생변수 생성 및 품질 검증을 포함한다.

In [1]:
import os
import re
import time
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

In [2]:
def find_sql(name):
    for base in (Path.cwd(), Path.cwd().parent):
        cand = base / 'sql' / name
        if cand.exists():
            return cand
    raise FileNotFoundError(f'sql/{name} 없음 (cwd={Path.cwd()})')

def load_queries(path):
    body = Path(path).read_text(encoding='utf-8')
    parts = re.split(r'(?m)^--\s*name:\s*(\w+).*$', body)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

Q = load_queries(find_sql('02_preprocessing_mart.sql'))

def run(name):
    return pd.read_sql(text(Q[name]), engine)

def execute(name):
    # -- 라인 주석 제거 후 세미콜론 분리 (주석 내 세미콜론 오분리 방지)
    body = re.sub(r'--[^\n]*', '', Q[name])
    with engine.begin() as conn:
        for stmt in [s for s in body.split(';') if s.strip()]:
            conn.execute(text(stmt))

## 1. `mart_session` 생성

분석 단위는 세션이다. 기존 세션 속성·카운트·revenue를 유지하고, 상품을 구분하지 않는 strict 순차 플래그를 추가한다. 반복 실행에서는 기존 mart_session·mart_session_product를 함께 사용한다. 생성 SQL이나 raw가 변경된 경우에는 `REBUILD_SESSION_MARTS`와 `RUN_SESSION_MART_VALIDATION`을 모두 `True`로 바꿔 두 마트를 순서대로 재생성·검증한다.

In [3]:
REBUILD_SESSION_MARTS = False  # 생성 SQL·raw 변경 시에만 True
RUN_SESSION_MART_VALIDATION = False  # 재생성·정합성 재확인 시에만 True

if REBUILD_SESSION_MARTS and not RUN_SESSION_MART_VALIDATION:
    raise ValueError('마트 재생성 시 RUN_SESSION_MART_VALIDATION도 True여야 합니다.')

if REBUILD_SESSION_MARTS:
    t0 = time.time()
    execute('create_mart_session')
    session_build_sec = time.time() - t0
    print(f"mart_session 재생성 완료 · {session_build_sec:.0f}s")
else:
    print('기존 mart_session 사용')

기존 mart_session 사용


## 2. `mart_session` 검증 a — 행수

In [4]:
if RUN_SESSION_MART_VALIDATION:
    mart_n = int(run('mart_rowcount').iloc[0, 0])
    raw_n = int(run('raw_valid_count').iloc[0, 0])
    print(f"마트 행수 = {mart_n:,}")
    print(f"raw 유효 세션 = {raw_n:,}")
    print(f"일치: {mart_n == raw_n}")
else:
    print('검증a 생략 · RUN_SESSION_MART_VALIDATION=False')

검증a 생략 · RUN_SESSION_MART_VALIDATION=False


## 3. `mart_session` 검증 b — 유형별 카운트

분석 단위는 이벤트다. 유효 세션 범위에서 `price >= 0`인 유형별 raw 건수와 세션 마트의 합을 대조한다.

In [5]:
if RUN_SESSION_MART_VALIDATION:
    mart_c = run('mart_type_counts').iloc[0]
    raw_c = run('raw_type_counts').set_index('event_type')['건수']
    order = ['view', 'cart', 'remove_from_cart', 'purchase']
    cmp_b = pd.DataFrame({
        'mart': [int(mart_c[t]) for t in order],
        'raw': [int(raw_c[t]) for t in order],
    }, index=order)
    cmp_b['일치'] = cmp_b['mart'] == cmp_b['raw']
    display(cmp_b)
else:
    print('검증b 생략 · RUN_SESSION_MART_VALIDATION=False')

검증b 생략 · RUN_SESSION_MART_VALIDATION=False


## 4. `mart_session` 검증 c — revenue

분석 단위는 세션 집계 금액이다. 유효 세션의 `purchase`·`price > 0` 합을 대조한다.

In [6]:
if RUN_SESSION_MART_VALIDATION:
    mart_r = float(run('mart_revenue').iloc[0, 0])
    raw_r = float(run('raw_revenue').iloc[0, 0])
    print(f"마트 revenue = {mart_r:,.2f}")
    print(f"raw revenue = {raw_r:,.2f}")
    print(f"일치: {round(mart_r, 2) == round(raw_r, 2)}")
else:
    print('검증c 생략 · RUN_SESSION_MART_VALIDATION=False')

검증c 생략 · RUN_SESSION_MART_VALIDATION=False


## 5. `mart_session` 검증 d — strict 순차 플래그

분석 단위는 세션이다. raw에서 독립 계산한 `view < cart`와 `view < cart < purchase` 도달 수를 마트 플래그 합과 대조한다.

In [7]:
if RUN_SESSION_MART_VALIDATION:
    raw_session_ordered = run('raw_ordered_counts').iloc[0]
    mart_session_ordered = run('mart_ordered_counts').iloc[0]
    session_order = ['view_units', 'strict_stage2', 'strict_stage3']
    cmp_session_ordered = pd.DataFrame({
        'mart': [int(mart_session_ordered[t]) for t in session_order],
        'raw': [int(raw_session_ordered[t]) for t in session_order],
    }, index=session_order)
    cmp_session_ordered['일치'] = cmp_session_ordered['mart'] == cmp_session_ordered['raw']
    display(cmp_session_ordered)
else:
    print('검증d 생략 · RUN_SESSION_MART_VALIDATION=False')

검증d 생략 · RUN_SESSION_MART_VALIDATION=False


> **동일 시각 민감도 실행 이력**
>
> 아래 셀의 strict·inclusive 수치는 `RUN_SESSION_MART_VALIDATION=True`로 실행한 검수 시점에 raw 독립 집계로 계산해 확인하고 `docs/metrics.md` §4에 기록한 값이다(세션 view→cart `710,008→710,651`, view→cart→purchase `97,600→97,628`). 이후 장시간 raw 재집계를 피하려고 기본값을 `False`로 두었고, 커밋된 출력의 `동일 시각 민감도 재계산 생략`은 검증 실패가 아니라 재집계를 건너뛴 상태를 뜻한다. 대표값은 strict이며 inclusive는 동일 시각 처리 민감도 확인용이다. 마트의 순서 정의나 원본 데이터가 바뀔 때만 스위치를 켜 재확인한다.

In [8]:
if RUN_SESSION_MART_VALIDATION:
    sv = int(raw_session_ordered['view_units'])
    ss2 = int(raw_session_ordered['strict_stage2'])
    si2 = int(raw_session_ordered['inclusive_stage2'])
    ss3 = int(raw_session_ordered['strict_stage3'])
    si3 = int(raw_session_ordered['inclusive_stage3'])
    session_sensitivity = pd.DataFrame({
        'strict(%)': [ss2 / sv * 100, ss3 / ss2 * 100, ss3 / sv * 100],
        'inclusive(%)': [si2 / sv * 100, si3 / si2 * 100, si3 / sv * 100],
    }, index=['view→cart', 'cart→purchase', '3단계 완주'])
    session_sensitivity['차이(%p)'] = session_sensitivity['inclusive(%)'] - session_sensitivity['strict(%)']
    print(f"동일 시각 포함 증가: stage2 {si2 - ss2:,}세션 · stage3 {si3 - ss3:,}세션")
    display(session_sensitivity.round(4))
else:
    print('동일 시각 민감도 재계산 생략')

동일 시각 민감도 재계산 생략


## 6. `mart_session` 검증 e — 플래그 논리 관계

In [9]:
if RUN_SESSION_MART_VALIDATION:
    session_logic = run('mart_ordered_logic').iloc[0].astype(int)
    display(session_logic)
else:
    print('검증e 생략 · RUN_SESSION_MART_VALIDATION=False')

검증e 생략 · RUN_SESSION_MART_VALIDATION=False


## 7. `mart_session` 검증 종합

In [10]:
if RUN_SESSION_MART_VALIDATION:
    check_a = mart_n == raw_n
    check_b = bool(cmp_b['일치'].all())
    check_c = round(mart_r, 2) == round(raw_r, 2)
    check_d = bool(cmp_session_ordered['일치'].all())
    check_e = bool((session_logic == 0).all())
    session_gate = check_a and check_b and check_c and check_d and check_e
    print(f"검증a(행수): {check_a}")
    print(f"검증b(유형별 카운트): {check_b}")
    print(f"검증c(revenue): {check_c}")
    print(f"검증d(strict 순차 플래그): {check_d}")
    print(f"검증e(플래그 논리 관계): {check_e}")
    print(f"mart_session 전체 통과: {session_gate}")
else:
    session_gate = None
    print('mart_session 검증 전체 생략 · 기존 검증 완료 마트 사용')

mart_session 검증 전체 생략 · 기존 검증 완료 마트 사용


## 8. `mart_session_product` 생성

분석 단위는 세션·상품(`user_session × product_id`)이다. 유효 세션 안에서 같은 상품의 도달 카운트와 strict 순차 플래그를 저장한다. cart-only·purchase-only 조합도 직접 진입 경로 분석을 위해 보존한다. 재생성 모드에서는 mart_session 검증 통과를 선행 조건으로 유지하고, 반복 실행 모드에서는 기존 테이블을 그대로 사용한다.

In [11]:
if REBUILD_SESSION_MARTS:
    if session_gate is not True:
        raise RuntimeError('mart_session 검증 실패: mart_session_product 생성을 중단합니다.')
    t0 = time.time()
    execute('create_mart_session_product')
    session_product_build_sec = time.time() - t0
    print(f"mart_session_product 재생성 완료 · {session_product_build_sec:.0f}s")
else:
    print('기존 mart_session_product 사용')

기존 mart_session_product 사용


## 9. `mart_session_product` 검증 a — 행수·복합키

In [12]:
if RUN_SESSION_MART_VALIDATION:
    msp_n = int(run('mart_session_product_count').iloc[0, 0])
    raw_msp_n = int(run('raw_session_product_count').iloc[0, 0])
    msp_pk_columns = int(run('mart_session_product_key').iloc[0, 0])
    print(f"마트 세션·상품 행수 = {msp_n:,}")
    print(f"raw 세션·상품 조합 = {raw_msp_n:,}")
    print(f"행수 일치: {msp_n == raw_msp_n}")
    print(f"복합 기본키 2열 확인: {msp_pk_columns == 2}")
else:
    print('검증a 생략 · RUN_SESSION_MART_VALIDATION=False')

검증a 생략 · RUN_SESSION_MART_VALIDATION=False


## 10. `mart_session_product` 검증 b — 유형별 카운트

분석 단위는 이벤트다. 유효 세션·상품 범위에서 `price >= 0`인 raw 건수와 마트의 합을 대조한다.

In [13]:
if RUN_SESSION_MART_VALIDATION:
    mart_msp_c = run('mart_session_product_type_counts').iloc[0]
    raw_msp_c = run('raw_session_product_type_counts').set_index('event_type')['건수']
    msp_order = ['view', 'cart', 'purchase']
    cmp_msp_type = pd.DataFrame({
        'mart': [int(mart_msp_c[t]) for t in msp_order],
        'raw': [int(raw_msp_c[t]) for t in msp_order],
    }, index=msp_order)
    cmp_msp_type['일치'] = cmp_msp_type['mart'] == cmp_msp_type['raw']
    display(cmp_msp_type)
else:
    print('검증b 생략 · RUN_SESSION_MART_VALIDATION=False')

검증b 생략 · RUN_SESSION_MART_VALIDATION=False


## 11. `mart_session_product` 검증 c — strict 순차 플래그

분석 단위는 세션·상품이다. raw에서 독립 계산한 동일 상품 순차 도달 수를 마트 플래그 합과 대조한다.

In [14]:
if RUN_SESSION_MART_VALIDATION:
    raw_msp_ordered = run('raw_session_product_ordered_counts').iloc[0]
    mart_msp_ordered = run('mart_session_product_ordered_counts').iloc[0]
    msp_stage_order = ['view_units', 'strict_stage2', 'strict_stage3']
    cmp_msp_ordered = pd.DataFrame({
        'mart': [int(mart_msp_ordered[t]) for t in msp_stage_order],
        'raw': [int(raw_msp_ordered[t]) for t in msp_stage_order],
    }, index=msp_stage_order)
    cmp_msp_ordered['일치'] = cmp_msp_ordered['mart'] == cmp_msp_ordered['raw']
    display(cmp_msp_ordered)
else:
    print('검증c 생략 · RUN_SESSION_MART_VALIDATION=False')

검증c 생략 · RUN_SESSION_MART_VALIDATION=False


> **동일 시각 민감도 실행 이력**
>
> 위 세션 민감도와 동일하게, 아래 세션·상품 strict·inclusive 수치도 `RUN_SESSION_MART_VALIDATION=True` 검수 시점에 확인해 `docs/metrics.md` §4에 기록한 값이다(동일 상품 view→cart `885,651→887,130`, view→cart→purchase `132,235→132,425`). 현재 `False`의 `동일 시각 민감도 재계산 생략`은 재집계를 건너뛴 상태이며, 대표값 strict은 그대로 사용하고 inclusive는 동일 시각 처리 민감도 확인용이다.

In [15]:
if RUN_SESSION_MART_VALIDATION:
    pv = int(raw_msp_ordered['view_units'])
    ps2 = int(raw_msp_ordered['strict_stage2'])
    pi2 = int(raw_msp_ordered['inclusive_stage2'])
    ps3 = int(raw_msp_ordered['strict_stage3'])
    pi3 = int(raw_msp_ordered['inclusive_stage3'])
    msp_sensitivity = pd.DataFrame({
        'strict(%)': [ps2 / pv * 100, ps3 / ps2 * 100, ps3 / pv * 100],
        'inclusive(%)': [pi2 / pv * 100, pi3 / pi2 * 100, pi3 / pv * 100],
    }, index=['view→cart', 'cart→purchase', '3단계 완주'])
    msp_sensitivity['차이(%p)'] = msp_sensitivity['inclusive(%)'] - msp_sensitivity['strict(%)']
    print(f"동일 시각 포함 증가: stage2 {pi2 - ps2:,}조합 · stage3 {pi3 - ps3:,}조합")
    display(msp_sensitivity.round(4))
else:
    print('동일 시각 민감도 재계산 생략')

동일 시각 민감도 재계산 생략


## 12. `mart_session_product` 검증 d — 플래그 논리 관계

In [16]:
if RUN_SESSION_MART_VALIDATION:
    msp_logic = run('mart_session_product_logic').iloc[0].astype(int)
    display(msp_logic)
else:
    print('검증d 생략 · RUN_SESSION_MART_VALIDATION=False')

검증d 생략 · RUN_SESSION_MART_VALIDATION=False


## 13. 검증 종합

In [17]:
if RUN_SESSION_MART_VALIDATION:
    check_msp_a = msp_n == raw_msp_n and msp_pk_columns == 2
    check_msp_b = bool(cmp_msp_type['일치'].all())
    check_msp_c = bool(cmp_msp_ordered['일치'].all())
    check_msp_d = bool((msp_logic == 0).all())
    msp_gate = check_msp_a and check_msp_b and check_msp_c and check_msp_d
    all_passed = session_gate and msp_gate
    print(f"mart_session 전체 통과: {session_gate}")
    print(f"검증a(행수·복합키): {check_msp_a}")
    print(f"검증b(유형별 카운트): {check_msp_b}")
    print(f"검증c(strict 순차 플래그): {check_msp_c}")
    print(f"검증d(플래그 논리 관계): {check_msp_d}")
    print(f"mart_session_product 전체 통과: {msp_gate}")
    print(f"02 전체 검증 통과: {all_passed}")
else:
    msp_gate = None
    all_passed = None
    print('02 raw 독립 검증 전체 생략 · 기존 검증 완료 마트 사용')

02 raw 독립 검증 전체 생략 · 기존 검증 완료 마트 사용


## 14. `mart_user_product_session` 생성

분석 단위는 유효 세션·상품(`user_session × product_id`)이다. 기존 두 마트와 달리 remove-only 조합도 포함하고 네 이벤트의 최초·최종 시각과 최초 구매 전 마지막 행동 시각을 보존한다. 06은 이 마트만 조회해 세션 경계를 넘는 구매 여정을 구성한다. 기존 마트와 별도 스위치로 관리하므로 이 마트만 재생성해도 `mart_session`·`mart_session_product`는 바뀌지 않는다.

In [18]:
REBUILD_JOURNEY_MART = False  # 여정 마트 생성 SQL·raw 변경 시에만 True
RUN_JOURNEY_MART_VALIDATION = False  # 재생성·정합성 재확인 시에만 True

if REBUILD_JOURNEY_MART and not RUN_JOURNEY_MART_VALIDATION:
    raise ValueError('여정 마트 재생성 시 RUN_JOURNEY_MART_VALIDATION도 True여야 합니다.')

if REBUILD_JOURNEY_MART:
    t0 = time.time()
    execute('create_mart_user_product_session')
    journey_build_sec = time.time() - t0
    print(f'mart_user_product_session 재생성 완료 · {journey_build_sec:.0f}s')
else:
    print('기존 mart_user_product_session 사용')

mart_user_product_session 재생성 완료 · 1833s


## 15. 구매 여정 마트 검증 a — raw 행수·카운트·시각·revenue

분석 단위는 세션·상품과 이벤트다. raw를 독립 집계해 마트의 전체 행수와 네 이벤트 합, 최초·최종 시각, revenue를 대조한다.

In [19]:
if RUN_JOURNEY_MART_VALIDATION:
    raw_journey = run('raw_user_product_session_reconciliation').iloc[0]
    mart_journey = run('mart_user_product_session_summary').iloc[0]
    journey_sum_pairs = [
        ('행수', 'raw_행수', '마트_행수'),
        ('view', 'raw_views', '마트_views'),
        ('cart', 'raw_carts', '마트_carts'),
        ('remove', 'raw_removes', '마트_removes'),
        ('purchase', 'raw_purchases', '마트_purchases'),
    ]
    cmp_journey = pd.DataFrame({
        'raw': [int(raw_journey[r]) for _, r, _ in journey_sum_pairs],
        'mart': [int(mart_journey[m]) for _, _, m in journey_sum_pairs],
    }, index=[label for label, _, _ in journey_sum_pairs])
    cmp_journey['일치'] = cmp_journey['raw'] == cmp_journey['mart']
    display(cmp_journey)
    print(f"revenue 일치: {round(float(raw_journey['raw_revenue']), 2) == round(float(mart_journey['마트_revenue']), 2)}")
    journey_raw_errors = raw_journey[[
        '마트누락_행수', 'user_id_불일치', '카운트_불일치', '시각_불일치', 'revenue_불일치'
    ]].astype(int)
    display(journey_raw_errors)
else:
    print('검증a 생략 · RUN_JOURNEY_MART_VALIDATION=False')

,raw,mart,일치
행수,13385787,13385787,True
view,9284031,9284031,True
cart,5502430,5502430,True
remove,3729756,3729756,True
purchase,1229578,1229578,True


revenue 일치: True


마트누락_행수        0
user_id_불일치    0
카운트_불일치        0
시각_불일치         0
revenue_불일치    0
Name: 0, dtype: int64

## 16. 구매 여정 마트 검증 b — 최초 구매 전 행동 시각

분석 단위는 purchase가 있는 세션·상품이다. 최초 purchase보다 앞선 마지막 view·cart·remove 시각을 raw에서 다시 계산해 대조한다. 동일 시각은 strict 기준에서 제외한다.

In [20]:
if RUN_JOURNEY_MART_VALIDATION:
    before_purchase_check = run('raw_user_product_before_purchase_reconciliation').iloc[0]
    print(f"raw 구매 세션·상품 = {int(before_purchase_check['raw_구매전행동_행수']):,}")
    print(f"마트 구매 세션·상품 = {int(mart_journey['마트_구매행수']):,}")
    print(f"구매 전 행동 시각 불일치 = {int(before_purchase_check['구매전행동시각_불일치']):,}")
else:
    print('검증b 생략 · RUN_JOURNEY_MART_VALIDATION=False')

raw 구매 세션·상품 = 1,221,747
마트 구매 세션·상품 = 1,221,747
구매 전 행동 시각 불일치 = 0


## 17. 구매 여정 마트 검증 c — 기존 순차 플래그

remove-only 조합을 추가하더라도 기존 `mart_session_product`의 strict 순차 플래그 합은 그대로 보존되어야 한다.

In [21]:
if RUN_JOURNEY_MART_VALIDATION:
    journey_flags = run('mart_user_product_session_flags').iloc[0].astype(int)
    display(journey_flags)
else:
    print('검증c 생략 · RUN_JOURNEY_MART_VALIDATION=False')

여정마트_stage2    885651
기존마트_stage2    885651
여정마트_stage3    132235
기존마트_stage3    132235
Name: 0, dtype: int64

## 18. 구매 여정 마트 검증 d — 논리 관계·키

카운트와 시각의 NULL 관계, 최초·최종 시각 순서, 구매 전 시각의 strict 조건, 플래그 포함 관계와 인덱스를 확인한다.

In [22]:
if RUN_JOURNEY_MART_VALIDATION:
    journey_logic = run('mart_user_product_session_logic').iloc[0].astype(int)
    journey_key = run('mart_user_product_session_key').iloc[0].astype(int)
    display(journey_logic)
    display(journey_key)
else:
    print('검증d 생략 · RUN_JOURNEY_MART_VALIDATION=False')

view_시각오류        0
cart_시각오류        0
remove_시각오류      0
purchase_시각오류    0
최초최종_역전          0
구매전시각_오류         0
플래그값_오류          0
순차포함관계_오류        0
최초구매경로_오류        0
Name: 0, dtype: int64

기본키_열수           2
사용자상품시각_인덱스열수    4
상품시각_인덱스열수       2
구매시각_인덱스열수       1
Name: 0, dtype: int64

## 19. 구매 여정 마트 검증 종합

In [23]:
if RUN_JOURNEY_MART_VALIDATION:
    check_journey_a = (
        bool(cmp_journey['일치'].all())
        and round(float(raw_journey['raw_revenue']), 2) == round(float(mart_journey['마트_revenue']), 2)
        and bool((journey_raw_errors == 0).all())
    )
    check_journey_b = (
        int(before_purchase_check['raw_구매전행동_행수']) == int(mart_journey['마트_구매행수'])
        and int(before_purchase_check['구매전행동시각_불일치']) == 0
    )
    check_journey_c = (
        int(journey_flags['여정마트_stage2']) == int(journey_flags['기존마트_stage2'])
        and int(journey_flags['여정마트_stage3']) == int(journey_flags['기존마트_stage3'])
    )
    check_journey_d = (
        bool((journey_logic == 0).all())
        and journey_key.to_dict() == {
            '기본키_열수': 2,
            '사용자상품시각_인덱스열수': 4,
            '상품시각_인덱스열수': 2,
            '구매시각_인덱스열수': 1,
        }
    )
    journey_gate = check_journey_a and check_journey_b and check_journey_c and check_journey_d
    print(f'검증a(raw 행수·카운트·시각·revenue): {check_journey_a}')
    print(f'검증b(최초 구매 전 행동 시각): {check_journey_b}')
    print(f'검증c(기존 strict 플래그 보존): {check_journey_c}')
    print(f'검증d(논리 관계·키): {check_journey_d}')
    print(f'mart_user_product_session 전체 통과: {journey_gate}')
    if not journey_gate:
        raise RuntimeError('구매 여정 마트 검증 실패: 06 분석으로 진행할 수 없습니다.')
else:
    journey_gate = None
    print('구매 여정 마트 검증 전체 생략 · 기존 검증 완료 마트 사용')

검증a(raw 행수·카운트·시각·revenue): True
검증b(최초 구매 전 행동 시각): True
검증c(기존 strict 플래그 보존): True
검증d(논리 관계·키): True
mart_user_product_session 전체 통과: True
